# Qwen2.5-Coder-7B QLoRA Fine-tuning (Unsloth) — v9 (ChatML)

Trains `Qwen/Qwen2.5-Coder-7B` with QLoRA using **Unsloth** on `chatml_v9.jsonl`.

**Dataset:** `chatml_v9.jsonl` — 11,508 rows (= `chatml_v7.jsonl` without its 12 duplicate rows), single column `text`, formatted as Qwen ChatML (`<|im_start|>user ... <|im_end|>\n<|im_start|>assistant ... <|im_end|>`).

**Difference from v7:** every assistant turn now ends with ```` ```\n<|im_end|> ```` instead of ```` ```<|im_end|> ````, so that EvalPlus' stop string `\n```\n` also matches and generation no longer depends only on a weakly trained `<|im_end|>` (v7/v8 produced junk tokens after the closing fence in ~35% of HumanEval answers). Hyper-parameters are kept identical to v7/v8, except `EMBED_MODE = "lm_head"` (LoRA on `lm_head` so the `<|im_end|>` logit is learned too); set `EMBED_MODE = "none"` for a data-only comparison against v7.

In [ ]:
!pip install -q -U unsloth unsloth_zoo bitsandbytes
import importlib.metadata as md
for p in ["unsloth", "unsloth_zoo", "bitsandbytes", "trl", "peft", "transformers", "accelerate", "torch"]:
    print(f"{p:14s}", md.version(p))

## 1. Imports & Login

In [ ]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")   # giảm phân mảnh VRAM; phải đặt trước khi import torch
import glob
import json
import torch
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
from datasets import load_dataset, Dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(hf_token, add_to_git_credential=True)

REPORT_TO = "none"

BASE_MODEL = "Qwen/Qwen2.5-Coder-7B"
HUB_MODEL_NAME = "hoangnh39/qwen2.5-coder-7b-opencode-v9.2"   # lặp lại v9.1 với seed khác

NUM_EPOCHS = 1
LEARNING_RATE = 1e-4
LORA_R = 32
LORA_ALPHA = 32       
SEED = 1234                # v9.1 dùng 3407; đổi seed = đổi khởi tạo LoRA, thứ tự dữ liệu và tập val

# Học token ChatML (<|im_start|>/<|im_end|>) mà Qwen base chưa train kỹ, qua embed_tokens + lm_head:
#   "lm_head": LoRA hạng thấp chỉ trên lm_head (phía sinh token: logit của <|im_end|>) -> chạy được trên T4 (đã thử)
#   "lora": LoRA trên cả embed_tokens và lm_head -> OOM trên T4 (đã thử): bước loss gộp của Unsloth cấp thêm bộ đệm fp32 2,03 GiB (cỡ ma trận lm_head) để cộng dồn gradient
#   "full": train đầy đủ (~545M tham số mỗi ma trận, fp32 trên T4 vì không có bf16) -> OOM trên T4, chỉ dùng cho GPU >= 24GB
#   "none": không học, chỉ dựa vào dừng theo chuỗi khi eval -> chạy được (đã thử); dùng để so riêng tác dụng của dữ liệu v9
# Nếu "lm_head" cũng bị OOM: (1) per_device_train_batch_size=1 + gradient_accumulation_steps=16 (cell Train, batch hiệu dụng vẫn 16),
#             (2) thêm device_map="unsloth_balanced" vào from_pretrained để chia model qua 2 GPU T4, (3) đặt EMBED_MODE = "none".
EMBED_MODE = "none"       # giữ đúng cấu hình v9.1 (chỉ đổi seed)
EMBED_LR = 1e-5           # Unsloth khuyến nghị thấp hơn lr của LoRA 2-10 lần

VAL_FRACTION = 0.03
EVAL_STEPS = 200
MAX_VAL_SAMPLES = 300      # eval ít mẫu hơn để mỗi lần eval không tốn nhiều thời gian

HF_DATASET_ID = None
DATASET_FILENAME = "chatml_v9.jsonl"

MAX_SEQ_LENGTH = 512
MAX_TRAIN_SAMPLES = None

## 2. Load Base Model + Attach LoRA (Unsloth)

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,          # auto-detect bf16/fp16 support
    load_in_4bit=True,
)

target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
if EMBED_MODE == "lora":
    target_modules = target_modules + ["embed_tokens", "lm_head"]      # LoRA hạng thấp trên cả hai ma trận
elif EMBED_MODE == "lm_head":
    target_modules = target_modules + ["lm_head"]                      # LoRA hạng thấp chỉ trên lm_head, ít VRAM nhất
assert EMBED_MODE in ("none", "lm_head", "lora", "full"), EMBED_MODE
modules_to_save = ["embed_tokens", "lm_head"] if EMBED_MODE == "full" else None   # train đầy đủ: ~+4GB VRAM trở lên, OOM trên T4

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0,                       # 0 is required for Unsloth's fused kernels
    target_modules=target_modules,
    modules_to_save=modules_to_save,
    bias="none",
    use_gradient_checkpointing="unsloth",
    use_rslora=True,
    random_state=SEED,
)
model.print_trainable_parameters()
print("GPU mem:", f"{torch.cuda.memory_allocated()/1e9:.1f} GB")

## 3. Load Dataset (OpenCodeInstruct HQ 10K, ChatML)

In [ ]:
if HF_DATASET_ID:
    dataset = load_dataset(HF_DATASET_ID, split="train")
else:
    matches = glob.glob(f"/kaggle/input/**/{DATASET_FILENAME}", recursive=True)
    assert matches, f"Không thấy {DATASET_FILENAME} trong /kaggle/input — hãy attach Kaggle Dataset chứa file này."
    print("Using", matches[0])
    dataset = load_dataset("json", data_files=matches[0], split="train")

assert "text" in dataset.column_names, f"Cần cột 'text', hiện có: {dataset.column_names}"
print(f"Loaded {len(dataset):,} examples")
n_bad_end = sum(not t.endswith("```\n<|im_end|>") for t in dataset["text"])
assert n_bad_end == 0, f"{n_bad_end} rows do not end with a closing fence + newline + <|im_end|> (is this chatml_v9.jsonl?)"

def n_tokens(batch):
    return {"n_tok": [len(ids) for ids in tokenizer(batch["text"], add_special_tokens=False)["input_ids"]]}

dataset = dataset.map(n_tokens, batched=True, batch_size=1000)
before = len(dataset)
dataset = dataset.filter(lambda ex: ex["n_tok"] <= MAX_SEQ_LENGTH).remove_columns("n_tok")
print(f"Kept {len(dataset):,}/{before:,} samples with <= {MAX_SEQ_LENGTH} tokens")

dataset = dataset.shuffle(seed=SEED)
if MAX_TRAIN_SAMPLES:
    dataset = dataset.select(range(min(MAX_TRAIN_SAMPLES, len(dataset))))
split = dataset.train_test_split(test_size=VAL_FRACTION, seed=SEED)
train_ds = split["train"]
val_ds = split["test"].select(range(min(MAX_VAL_SAMPLES, len(split["test"]))))
print(f"Train {len(train_ds):,} | Val {len(val_ds):,}")
print(train_ds[0]["text"][:500])
if "<|im_start|>system" not in train_ds[0]["text"]:
    print("WARNING: dữ liệu không có system prompt, nhưng chat template của Qwen tự thêm khi eval -> nên xuất lại với --system-prompt")

## 4. Train

In [ ]:
from huggingface_hub import HfApi
from transformers import TrainerCallback
from unsloth import UnslothTrainer, UnslothTrainingArguments
from unsloth.chat_templates import train_on_responses_only

class PushEpochAdapter(TrainerCallback):
    """Sau mỗi epoch: lưu LoRA adapter và đẩy lên branch `epoch-N` của HUB_MODEL_NAME."""
    def __init__(self, tokenizer):
        self.api, self.tokenizer = HfApi(), tokenizer
        self.api.create_repo(HUB_MODEL_NAME, private=True, exist_ok=True)

    def on_epoch_end(self, args, state, control, model=None, **kwargs):
        epoch = int(round(state.epoch))
        branch, path = f"epoch-{epoch}", f"./adapter_epoch{epoch}"
        model.save_pretrained(path)
        self.tokenizer.save_pretrained(path)
        self.api.create_branch(HUB_MODEL_NAME, branch=branch, exist_ok=True)
        self.api.upload_folder(repo_id=HUB_MODEL_NAME, folder_path=path, revision=branch,
                               commit_message=f"LoRA adapter after epoch {epoch}")
        print(f"Pushed epoch {epoch} adapter -> {HUB_MODEL_NAME}@{branch}")

train_args = UnslothTrainingArguments(
    output_dir="./checkpoints",
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,    # effective batch size 16
    optim="adamw_8bit",
    learning_rate=LEARNING_RATE,
    embedding_learning_rate=EMBED_LR if EMBED_MODE == "full" else None,   # chỉ dùng cho train đầy đủ; LoRA (lm_head/lora) giữ learning_rate thường
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    weight_decay=0.01,
    max_grad_norm=0.3,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=EVAL_STEPS,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    push_to_hub=False,
    seed=SEED,
    data_seed=SEED,
    report_to=REPORT_TO,
)

trainer = UnslothTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    args=train_args,
    callbacks=[PushEpochAdapter(tokenizer)],
)

# Mask phần user: loss chỉ tính từ sau "<|im_start|>assistant\n"
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

# Kiểm tra mask: phần user phải là -100, phần assistant mới có label
ex = trainer.train_dataset[0]
print("Labels kept:", sum(l != -100 for l in ex["labels"]), "/", len(ex["labels"]))
print(tokenizer.decode([t for t, l in zip(ex["input_ids"], ex["labels"]) if l != -100])[:300])

trainer_stats = trainer.train()
print(trainer_stats)

## 5. Save / Push Adapter

In [ ]:
print("Best checkpoint:", trainer.state.best_model_checkpoint, "| best eval_loss:", trainer.state.best_metric)
model.push_to_hub(HUB_MODEL_NAME, private=True)
tokenizer.push_to_hub(HUB_MODEL_NAME, private=True)
print(f"Final adapter pushed to https://huggingface.co/{HUB_MODEL_NAME}")

## 6. Smoke Test (sanity check, not a benchmark)

In [ ]:
FastLanguageModel.for_inference(model)

# Smoke test (không phải benchmark): model có tự dừng bằng <|im_end|> ngay sau "```\n" không?
# Tokenizer của Qwen2.5-Coder-7B (base) không có chat_template nên apply_chat_template báo lỗi; dựng prompt ChatML bằng tay,
# giống hệt dữ liệu train (system prompt mặc định của Qwen), rồi prefill "```python\n" như EvalPlus.
SYSTEM = "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."
def chatml_prompt(task):
    return f"<|im_start|>system\n{SYSTEM}<|im_end|>\n<|im_start|>user\n{task}<|im_end|>\n<|im_start|>assistant\n"
tasks = [
    "Write a Python function `is_palindrome(s: str) -> bool` that checks whether a string is a palindrome, ignoring case and non-alphanumeric characters.",
    "Write a Python function `fib(n: int) -> int` that returns the n-th Fibonacci number.",
    "Write a Python function `flatten(lst: list) -> list` that flattens an arbitrarily nested list of integers.",
    "Write a Python function `count_vowels(s: str) -> int` that counts the vowels in a string, ignoring case.",
    "Write a Python function `two_sum(nums: list, target: int) -> list` that returns the indices of two numbers adding up to target.",
    "Write a Python function `is_prime(n: int) -> bool` that checks whether n is a prime number.",
    "Write a Python function `merge_sorted(a: list, b: list) -> list` that merges two sorted lists into one sorted list.",
    "Write a Python function `roman_to_int(s: str) -> int` that converts a roman numeral to an integer.",
]
im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
stopped, clean_end = 0, 0
for task in tasks:
    prompt = chatml_prompt(task) + "```python\n"
    inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(model.device)
    out = model.generate(**inputs, max_new_tokens=512, do_sample=False,
                         eos_token_id=[im_end_id, tokenizer.eos_token_id], pad_token_id=tokenizer.eos_token_id)
    gen = out[0][inputs["input_ids"].shape[1]:]
    text = tokenizer.decode(gen, skip_special_tokens=False)
    s = gen[-1].item() == im_end_id
    c = text.endswith("```\n<|im_end|>")
    stopped += s; clean_end += c
    print(f"stopped on <|im_end|>: {s} | ends with fence+newline: {c} | new tokens: {len(gen)} | {task[:50]}")
    if not c:
        print("   tail:", repr(text[-120:]))
print(f"\n---> stopped on <|im_end|>: {stopped}/{len(tasks)} | clean end: {clean_end}/{len(tasks)}  (mong đợi: {len(tasks)}/{len(tasks)})")